# 01 — Feature Engineering

S3 raw reviews + metadata → `feast materialize` (Spark + RAPIDS GPU) → Redis

- `user_features` — per-user aggregates (avg rating, review count, tenure)
- `item_features` — per-item review aggregates (avg rating, stddev, count)
- `item_metadata` — product catalog (title, brand, category, price) from metadata parquet

**Prerequisite:** Raw data in MinIO (`envsubst < manifests/data-download-job.yaml | oc apply -f -`)


In [ ]:
%pip install -q boto3 tabulate pandas pyarrow feast redis pyyaml s3fs kubernetes
%pip install yamlmagic --index-url https://pypi.org/simple
%load_ext yamlmagic

## Configuration


In [ ]:
%%yaml parameters

MATERIALIZE_START: "2020-01-01T00:00:00"
MATERIALIZE_END: "2026-12-31T23:59:59"

# Feast repo path inside the Feast pod (git-synced by FeatureStore CR)
FEAST_REPO: /feast-data/smartshop/feast/feature_repo


In [ ]:
from _config import *
globals().update(parameters)

validate()

---
## Pre-flight — Verify Raw Data in S3


In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"Existing buckets: {buckets}")

resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=5)
raw_files = resp.get("Contents", [])
if raw_files:
    print(f"\n✓ Raw reviews: {len(raw_files)}+ files in smartshop-raw/raw/reviews/")
else:
    print("\n✗ No raw data — run the download manifest first:")
    print("  envsubst < manifests/data-download-job.yaml | oc apply -f -")
    raise RuntimeError("Raw data missing")

expected_cats = ["Electronics", "Books", "Home_and_Kitchen"]
meta_resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/metadata/", MaxKeys=50)
meta_keys = [o["Key"] for o in meta_resp.get("Contents", [])]
found_cats = [c for c in expected_cats if any(c in k for k in meta_keys)]
missing_cats = [c for c in expected_cats if c not in found_cats]

print(f"✓ Metadata: {len(meta_keys)} files in smartshop-raw/raw/metadata/")
print(f"  Categories with metadata: {found_cats}")
if missing_cats:
    print(f"  ⚠ Missing metadata for: {missing_cats}")
    print("    Re-run download job with METADATA_ONLY=true to fetch these")

### S3 file listing (optional — expand for detail)


In [ ]:
from tabulate import tabulate

print("=== Reviews ===\n")
paginator = s3.get_paginator("list_objects_v2")
total_size = 0
total_files = 0
rows = []

for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/reviews/"):
    for obj in page.get("Contents", []):
        size_mb = obj["Size"] / (1024 * 1024)
        total_size += obj["Size"]
        total_files += 1
        if total_files <= 15:
            rows.append([obj["Key"], f"{size_mb:.1f} MB"])

try:
    print(tabulate(rows, headers=["Key", "Size"], tablefmt="simple"))
except:
    for r in rows:
        print(f"  {r[0]:60s} {r[1]}")

if total_files > 15:
    print(f"  ... and {total_files - 15} more files")
print(f"\nTotal: {total_files} files, {total_size / (1024**3):.2f} GB")

print("\n=== Metadata ===\n")
meta_files = 0
meta_size = 0
for page in paginator.paginate(Bucket="smartshop-raw", Prefix="raw/metadata/"):
    for obj in page.get("Contents", []):
        meta_files += 1
        meta_size += obj["Size"]
        if meta_files <= 10:
            print(f"  {obj['Key']:60s} {obj['Size'] / (1024*1024):.1f} MB")
print(f"\nTotal metadata: {meta_files} files, {meta_size / (1024**3):.2f} GB")

In [ ]:
# Review schema preview
import pandas as pd, io
resp = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=1)
if resp.get("Contents"):
    obj = s3.get_object(Bucket="smartshop-raw", Key=resp["Contents"][0]["Key"])
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    print(f"Schema ({len(df):,} rows): {list(df.columns)}")
    display(df.head(2)) if hasattr(__builtins__, '__IPYTHON__') else print(df.head(2).to_string())

---
## Phase 1 — Register & Materialize

### 1.1 Register feature views (`store.apply`)


In [ ]:
import sys, os
from feast import FeatureStore

sys.path.insert(0, os.path.join(os.getcwd(), "feature_repo"))
from features import (
    user, item, raw_reviews_source, raw_metadata_source,
    user_features, item_features, item_metadata,
)

store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)

store.apply([
    user, item,
    raw_reviews_source, raw_metadata_source,
    user_features, item_features, item_metadata,
])

print("Registered entities:")
for e in store.list_entities():
    print(f"  {e.name}")
print("Registered feature views:")
for fv in store.list_feature_views():
    print(f"  {fv.name} ({len(fv.features)} features)")
print("✓ feast apply complete")

### 1.2 Materialize → Redis (Spark + RAPIDS on Feast pod)

In [ ]:
import time as _time
import redis as _redis
from kubernetes import client, config
from kubernetes.stream import stream

config.load_incluster_config()
v1 = client.CoreV1Api()

feast_pods = v1.list_namespaced_pod(
    NAMESPACE, label_selector="feast.dev/name=smartshop-feast",
    field_selector="status.phase=Running",
).items
feast_pod = feast_pods[0].metadata.name
print(f"Feast pod: {feast_pod}")

def _redis_ok():
    """Return True when Redis is healthy and not loading."""
    try:
        r = _redis.Redis(host=REDIS_HOST, port=REDIS_PORT,
                         password=REDIS_PASSWORD or None, socket_timeout=5)
        r.ping()
        return True
    except _redis.BusyLoadingError:
        return False
    except Exception:
        return False

def _wait_redis(label="Redis", max_wait=60):
    for i in range(max_wait // 5):
        if _redis_ok():
            return True
        print(f"  ⏳ {label}: waiting for Redis to stabilize ({(i+1)*5}s)...")
        _time.sleep(5)
    return False

def _materialize_view(view_name: str, attempt=1, max_attempts=2):
    """Materialize a single feature view with retry."""
    cmd = [
        "feast", "-c", FEAST_REPO,
        "materialize", MATERIALIZE_START, MATERIALIZE_END,
        "-v", view_name,
    ]
    print(f"\n{'='*60}")
    print(f"▶ Materializing: {view_name} (attempt {attempt}/{max_attempts})")
    print(f"  {' '.join(cmd)}")
    t0 = _time.time()

    try:
        resp = stream(
            v1.connect_get_namespaced_pod_exec,
            feast_pod, NAMESPACE, container="offline",
            command=cmd, stderr=True, stdout=True, stdin=False,
            _request_timeout=900,
        )
        elapsed = _time.time() - t0
        lines = resp.strip().split("\n")

        has_error = any(kw in resp.lower() for kw in ["error", "exception", "traceback", "failed"])
        for line in lines[-15:]:
            print(f"  {line}")

        if has_error and attempt < max_attempts:
            print(f"\n  ⚠ {view_name} had errors — retrying after Redis cooldown...")
            _wait_redis(view_name)
            return _materialize_view(view_name, attempt + 1, max_attempts)

        print(f"  ✓ {view_name} completed in {elapsed:.0f}s")
        return True
    except Exception as e:
        print(f"  ✗ {view_name} failed: {e}")
        if attempt < max_attempts:
            print(f"  Retrying after cooldown...")
            _wait_redis(view_name)
            return _materialize_view(view_name, attempt + 1, max_attempts)
        return False

# Materialize each view sequentially — avoids Redis pressure spikes
feature_views = ["user_features", "item_features", "item_metadata"]
results = {}

for fv in feature_views:
    _wait_redis(fv)
    results[fv] = _materialize_view(fv)

print(f"\n{'='*60}")
print("Summary:")
for fv, ok in results.items():
    print(f"  {'✓' if ok else '✗'} {fv}")

failed = [fv for fv, ok in results.items() if not ok]
if failed:
    print(f"\n⚠ Failed views: {failed} — re-run this cell to retry")
else:
    print("\n✓ All feature views materialized to Redis")

---
## Phase 2 — Verify


In [ ]:
import redis, struct
import pandas as pd

print("Feast registry:")
for fv in store.list_feature_views():
    features = [f.name for f in fv.features] if hasattr(fv, "features") else []
    print(f"  {fv.name:25s} ({len(features)} features)")

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD or None, decode_responses=False)
print(f"\nRedis keys: {r.dbsize():,}")

# Spot-check: verify each feature view has data by scanning a few keys
for entity, fv_name in [("user_id", "user_features"), ("item_id", "item_features"), ("item_id", "item_metadata")]:
    cursor, keys = r.scan(0, match=f"*{entity}*smartshop".encode(), count=50)
    has_fv = 0
    for k in keys[:20]:
        fields = r.hgetall(k)
        if any(fv_name.encode() in f for f in fields.keys()):
            has_fv += 1
    status = "✓" if has_fv > 0 else "✗ NOT FOUND"
    print(f"  {fv_name:25s} → Redis {status} ({has_fv}/{min(20, len(keys))} keys checked)")


### 2.1 User features


In [ ]:
import pandas as pd, struct

_, raw_keys = r.scan(cursor=0, match=b"*user_id*smartshop", count=20)
user_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    pos = raw.find(b"user_id")
    if pos < 0:
        continue
    offset = pos + len(b"user_id") + 4
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        user_ids.append(val)

sample_users = [{"user_id": uid} for uid in user_ids[:5]]
result = store.get_online_features(
    features=[
        "user_features:user_avg_rating",
        "user_features:user_review_count",
        "user_features:user_unique_items",
        "user_features:user_tenure_days",
    ],
    entity_rows=sample_users,
).to_dict()

df = pd.DataFrame(result)
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

### 2.2 Item features (with product metadata)


In [ ]:
_, raw_keys = r.scan(cursor=0, match=b"*item_id*smartshop", count=200)
item_ids = []
for rk in raw_keys:
    raw = rk if isinstance(rk, bytes) else rk.encode()
    pos = raw.find(b"item_id")
    if pos < 0:
        continue
    offset = pos + len(b"item_id") + 4
    if offset + 4 > len(raw):
        continue
    val_len = struct.unpack("<I", raw[offset:offset + 4])[0]
    val = raw[offset + 4 : offset + 4 + val_len].decode("utf-8", errors="ignore")
    if val:
        item_ids.append(val)

sample_items = [{"item_id": iid} for iid in item_ids[:5]]
result = store.get_online_features(
    features=[
        "item_metadata:item_title",
        "item_metadata:item_brand",
        "item_metadata:item_category",
        "item_features:item_avg_rating",
        "item_metadata:item_price",
        "item_features:item_review_count",
    ],
    entity_rows=sample_items,
).to_dict()

df = pd.DataFrame(result)
has_meta = df["item_title"].notna().sum()
print(f"{has_meta}/{len(df)} items have product metadata")
display(df) if hasattr(__builtins__, '__IPYTHON__') else print(df.to_string())

---
## Done

35M+ raw reviews + 2M product metadata → user features, item features, and item metadata in Redis via Feast + Spark + RAPIDS.

**Next →** `02_training.ipynb`
